# Monitoramento multimodal de pacientes — notebook principal (Google Colab)

Tech Challenge Fase 4 · FIAP IA para Devs. Este notebook é o **`main`** do projeto: prepara o ambiente,
lê as credenciais, (opcionalmente) treina o detector de instrumentos cirúrgicos com o CholecT50 e executa
o pipeline multimodal (vídeo + áudio + sinais vitais/prescrições → fusão → alerta).

**Como usar**
1. *Ambiente de execução → Alterar tipo → GPU (T4)* — só necessário para treinar o YOLOv8 (seção 4).
2. Cadastre as chaves Azure em **Secrets** (ícone 🔑 na barra lateral) e ative *Acesso ao notebook*:
   `AZURE_SPEECH_KEY`, `AZURE_SPEECH_REGION`, `AZURE_LANGUAGE_ENDPOINT`, `AZURE_LANGUAGE_KEY`
   (e, opcionais, `AZURE_EVENTGRID_ENDPOINT`, `AZURE_EVENTGRID_KEY`, `TEAMS_WEBHOOK_URL`).
3. Ajuste os **parâmetros** (seção 3) e execute as células em ordem. Etapas sem arquivo/credencial são puladas.

> O armazenamento do Colab é temporário: salve os resultados e os pesos no Google Drive (seção 7).

## 1. Código do projeto e dependências

O notebook **clona o repositório do GitHub** sozinho: só é preciso preencher `REPO_URL` na célula abaixo.
Se o repositório for **privado**, crie no Colab (ícone 🔑 *Segredos*) o segredo `GITHUB_TOKEN` com um token de
leitura (fine-grained, permissão *Contents: read*) e ative *Acesso ao notebook*. Ao reexecutar, o código é atualizado
com `git pull`.


In [ ]:
import os, sys, shutil, subprocess, zipfile, pathlib, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# --- Código do projeto: o notebook clona o repositório do GitHub (não é preciso enviar nenhum arquivo) ---
REPO_URL = "https://github.com/SEU-USUARIO/multimodal-health-monitor.git"   # <- troque pelo URL do seu repositório
REPO_BRANCH = None                  # None = branch padrão do repositório (ex.: "main")
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # só para repositório PRIVADO: segredo do Colab (ícone 🔑) com um token de leitura

REPO_NAME = REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]


def _git(*args, cwd=None, mask=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).strip()
        raise RuntimeError(f"git {args[0]} falhou: " + (msg.replace(mask, "***") if mask else msg))
    return r.stdout.strip()


def _github_token():
    try:
        from google.colab import userdata
        return userdata.get(GITHUB_TOKEN_SECRET) or None
    except Exception:           # segredo inexistente ou sem acesso ao notebook
        return None


if IN_COLAB:
    PROJECT_DIR = f"/content/{REPO_NAME}"
    if "SEU-USUARIO" in REPO_URL:
        raise ValueError("Defina REPO_URL com o URL do seu repositório no GitHub (célula acima).")
    token = _github_token()
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1) if token else REPO_URL
    branch_args = ["--branch", REPO_BRANCH] if REPO_BRANCH else []
    if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
        try:
            _git("clone", "--depth", "1", *branch_args, url, PROJECT_DIR, mask=token)
        except RuntimeError as e:
            raise RuntimeError(f"{e}\nSe o repositório for PRIVADO, crie o segredo {GITHUB_TOKEN_SECRET} no Colab "
                               "(ícone 🔑), com um token de leitura, e ative 'Acesso ao notebook'.") from None
        if token:                                      # não deixa o token gravado no .git/config
            _git("remote", "set-url", "origin", REPO_URL, cwd=PROJECT_DIR)
    else:                                              # sessão já tem o código: traz as atualizações
        try:
            _git("pull", "--ff-only", url, *([REPO_BRANCH] if REPO_BRANCH else ["HEAD"]), cwd=PROJECT_DIR, mask=token)
        except RuntimeError as e:
            print("Aviso: não consegui atualizar o código; usando a versão que já está na sessão.\n", e)
else:   # local: raiz do repositório (ou a pasta pai, se o Jupyter abriu em notebooks/)
    PROJECT_DIR = os.getcwd() if os.path.isdir("src") else os.path.dirname(os.getcwd())

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
assert os.path.isdir("src"), f"Pasta src/ não encontrada em {os.getcwd()} — confira REPO_URL."
try:
    print("Projeto em:", os.getcwd(), "| Colab:", IN_COLAB, "| commit:", _git("rev-parse", "--short", "HEAD"))
except Exception:
    print("Projeto em:", os.getcwd(), "| Colab:", IN_COLAB)

In [ ]:
# No Colab já existem torch, opencv, pandas, scikit-learn e matplotlib; instalamos o restante.
# libasound2: o SDK do Azure Speech usa ALSA no Linux.
if IN_COLAB:
    %pip install -q ultralytics python-dotenv pyyaml scipy librosa soundfile azure-cognitiveservices-speech azure-ai-textanalytics azure-eventgrid azure-core
    subprocess.run("apt-get -qq install -y libasound2 > /dev/null 2>&1 || true", shell=True)
else:
    print("Fora do Colab: instale com `pip install -r requirements.txt`.")

## 2. Credenciais Azure

As variáveis precisam ser definidas **antes** de importar `src` (o `src/config.py` as lê na importação).
Se você alterar uma chave depois, reinicie a sessão (*Ambiente de execução → Reiniciar sessão*).

In [ ]:
KEYS = ["AZURE_SPEECH_KEY", "AZURE_SPEECH_REGION", "AZURE_LANGUAGE_ENDPOINT", "AZURE_LANGUAGE_KEY",
        "AZURE_EVENTGRID_ENDPOINT", "AZURE_EVENTGRID_KEY", "TEAMS_WEBHOOK_URL"]


def _secret(name):
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:          # segredo inexistente ou sem acesso ao notebook
            return None
    return os.environ.get(name)


for k in KEYS:
    v = _secret(k)
    if v:
        os.environ[k] = v
os.environ.setdefault("AZURE_SPEECH_REGION", "brazilsouth")
if "src.config" in sys.modules:
    print("AVISO: src já foi importado nesta sessão; reinicie a sessão para aplicar chaves novas.")
print({k: ("ok" if os.environ.get(k) else "—") for k in KEYS})

## 3. Parâmetros da execução

In [ ]:
PATIENT_ID = "P001"

# Arquivos de entrada (None = pular a etapa). Cole os caminhos ou use UPLOAD_FILES = True.
DATA_DIR = "data/raw"
VIDEO_PHYSIO = None        # fisioterapia -> pose, ex.: "data/raw/fisio.mp4"
VIDEO_SURGERY = None       # cirurgia laparoscópica -> instrumentos, ex.: "data/raw/cirurgia.mp4"
AUDIO_FILE = None          # consulta, ex.: "data/raw/consulta.wav" (mp3/m4a são convertidos)

# Sinais vitais e prescrições (sintéticos por padrão; troque pelos seus CSVs se tiver dados)
USE_SYNTHETIC_DATA = True
VITALS_CSV = "data/samples/vitals_P001.csv"            # colunas: timestamp, hr, sbp, dbp, spo2
PRESCRIPTIONS_CSV = "data/samples/prescriptions_P001.csv"  # colunas: timestamp, drug, dose_mg

DISPATCH_ALERTS = True     # False = não grava/envia o alerta
UPLOAD_FILES = False       # True = abre o seletor de arquivos do Colab e salva em DATA_DIR

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

if UPLOAD_FILES and IN_COLAB:
    from google.colab import files
    for name in files.upload():
        shutil.move(name, os.path.join(DATA_DIR, name))
        print("salvo em", os.path.join(DATA_DIR, name))

if USE_SYNTHETIC_DATA:
    subprocess.run([sys.executable, "scripts/generate_synthetic_data.py"], check=True)


def to_wav16k(path):
    """Azure Speech: WAV PCM 16 kHz mono."""
    if not shutil.which("ffmpeg"):
        print("ffmpeg não encontrado; usando o arquivo como está.")
        return path
    out = str(pathlib.Path(path).with_suffix(".16k.wav"))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(path), "-ar", "16000", "-ac", "1", out],
                   check=True)
    return out


if AUDIO_FILE:
    AUDIO_FILE = to_wav16k(AUDIO_FILE)
if VIDEO_PHYSIO and VIDEO_SURGERY:
    print("AVISO: o pipeline analisa um vídeo por execução; será usado VIDEO_SURGERY. "
          "Para analisar o outro, rode a seção 6 novamente trocando os parâmetros.")
print("Entradas:", {"fisio": VIDEO_PHYSIO, "cirurgia": VIDEO_SURGERY, "audio": AUDIO_FILE,
                    "vitais": VITALS_CSV, "prescricoes": PRESCRIPTIONS_CSV})

In [ ]:
import json, logging
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from src.config import settings


def findings_table(findings):
    rows = [f.to_dict() for f in findings]
    cols = ["modality", "kind", "score", "description"]
    return pd.DataFrame(rows)[cols].sort_values("score", ascending=False) if rows else pd.DataFrame(columns=cols)

## 4. (Opcional) Detector de instrumentos cirúrgicos — CholecT50 + YOLOv8

Só é necessário se for analisar vídeo cirúrgico (`VIDEO_SURGERY`). O CholecT50 precisa ser obtido pelo
repositório oficial (licença CC BY-NC-SA 4.0) e colocado no Drive. Detalhes e limitações (caixas de
instrumento só em 5 vídeos) em `docs/cholect50.md`.

In [ ]:
RUN_CHOLECT50_PREP = False                          # converter o dataset para o formato YOLO
RUN_TRAIN = False                                   # treinar o YOLOv8
USE_DRIVE = False                                   # True: monta o Drive e reaproveita pesos já salvos
CHOLECT50_DIR = "/content/drive/MyDrive/CholecT50"  # pasta com videos/ e labels/
DRIVE_MODELS_DIR = "/content/drive/MyDrive/modelos_tech_challenge"  # onde guardar os pesos
EPOCHS = 60
BASE_MODEL = "yolov8n.pt"                           # yolov8s.pt = mais preciso e mais lento
IMGSZ, BATCH, WORKERS = 640, 16, 2                  # se faltar memória: IMGSZ=480 ou BATCH=8

import torch
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo de treino:", "GPU" if DEVICE == 0 else "CPU (lento — ative a GPU no Colab)")

WEIGHTS = "models/cholect50_tools.pt"
if IN_COLAB and (USE_DRIVE or RUN_CHOLECT50_PREP or RUN_TRAIN):
    if not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    saved = os.path.join(DRIVE_MODELS_DIR, "cholect50_tools.pt")
    if os.path.exists(saved) and not os.path.exists(WEIGHTS):   # reaproveita pesos já treinados
        os.makedirs("models", exist_ok=True)
        shutil.copy2(saved, WEIGHTS)
        print("Pesos restaurados do Drive:", saved)
print("Pesos cirúrgicos disponíveis:", os.path.exists(WEIGHTS))

In [ ]:
if RUN_CHOLECT50_PREP:
    # 1) estatísticas (nada é escrito) — confira quais vídeos têm caixas e quantas por classe
    !python scripts/prepare_cholect50_yolo.py --src "{CHOLECT50_DIR}" --out data/cholect50_yolo --dry-run

In [ ]:
if RUN_CHOLECT50_PREP:
    # 2) gera o dataset YOLO (copia as imagens para o disco local do Colab)
    !python scripts/prepare_cholect50_yolo.py --src "{CHOLECT50_DIR}" --out data/cholect50_yolo

In [ ]:
if RUN_TRAIN:
    !python scripts/train_yolo_cholect50.py --data data/cholect50_yolo/cholect50_yolo.yaml --model {BASE_MODEL} --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --workers {WORKERS} --device {DEVICE} --no-plots

In [ ]:
if RUN_TRAIN and os.path.exists(WEIGHTS):
    import json
    metrics = pathlib.Path("models/test_metrics.json")
    if metrics.exists():
        print(json.dumps(json.loads(metrics.read_text()), indent=2, ensure_ascii=False))
    if IN_COLAB:                                    # guarda os pesos no Drive
        os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
        for f in (WEIGHTS, str(metrics)):
            if os.path.exists(f):
                shutil.copy2(f, DRIVE_MODELS_DIR)
        print("Pesos e métricas salvos em", DRIVE_MODELS_DIR)

## 5. Etapas isoladas (para explorar e para a demonstração)

Cada célula roda um módulo sozinho e mostra o resultado. A seção 6 executa tudo junto, com fusão e alerta.

### 5.1 Sinais vitais e prescrições

In [ ]:
from src.anomaly.vitals import LIMITS, VITAL_COLS, VitalsAnomalyDetector
from src.anomaly.prescriptions import PrescriptionAnomalyDetector

vitals = pd.read_csv(VITALS_CSV, parse_dates=["timestamp"])
vit_findings = VitalsAnomalyDetector().detect(vitals, PATIENT_ID)

fig, axes = plt.subplots(len(VITAL_COLS), 1, figsize=(11, 8), sharex=True)
for ax, col in zip(axes, VITAL_COLS):
    lo, hi = LIMITS[col]
    ax.plot(vitals["timestamp"], vitals[col], lw=1)
    ax.axhspan(lo, hi, alpha=0.08)
    out = vitals[(vitals[col] < lo) | (vitals[col] > hi)]
    ax.scatter(out["timestamp"], out[col], color="red", s=18, zorder=3)
    ax.set_ylabel(col)
fig.suptitle(f"Sinais vitais — {PATIENT_ID} (faixa sombreada = limite de triagem; vermelho = fora do limite)")
plt.tight_layout(); plt.show()

display(findings_table(vit_findings))

In [ ]:
if PRESCRIPTIONS_CSV and os.path.exists(PRESCRIPTIONS_CSV):
    presc = pd.read_csv(PRESCRIPTIONS_CSV, parse_dates=["timestamp"])
    display(findings_table(PrescriptionAnomalyDetector().detect(presc, PATIENT_ID)))

### 5.2 Áudio (Azure Speech + Azure Language + voz)

In [ ]:
if AUDIO_FILE and os.environ.get("AZURE_SPEECH_KEY"):
    from src.audio.transcriber import SpeechTranscriber
    from src.audio.text_analytics import TextAnalyzer
    from src.audio.voice_features import VoiceAnalyzer

    transcript = SpeechTranscriber().transcribe(AUDIO_FILE)
    print("Transcrição:\n", transcript or "(vazia)")
    text_f = TextAnalyzer().analyze(transcript, PATIENT_ID, use_azure=bool(os.environ.get("AZURE_LANGUAGE_KEY")))
    feats, voice_f = VoiceAnalyzer().analyze(AUDIO_FILE, PATIENT_ID, transcript)
    display(pd.DataFrame([feats]))
    display(findings_table(text_f + voice_f))
else:
    print("Etapa pulada: defina AUDIO_FILE e a chave AZURE_SPEECH_KEY (seções 2 e 3).")

### 5.3 Vídeo

In [ ]:
from src.video.report import generate_video_report

if VIDEO_SURGERY:
    if not os.path.exists(settings.surgical_model):
        print("Pesos cirúrgicos não encontrados:", settings.surgical_model, "→ treine na seção 4.")
    else:
        from src.video.surgical_tools import SurgicalToolAnalyzer
        res = SurgicalToolAnalyzer().analyze(VIDEO_SURGERY, PATIENT_ID)
        print(res.summary)
        display(findings_table(res.findings))
        rep = generate_video_report(PATIENT_ID, VIDEO_SURGERY, res.findings, summary=res.summary)
        display(HTML(rep.read_text(encoding="utf-8")))
elif VIDEO_PHYSIO:
    from src.video.pose_analyzer import PoseAnalyzer
    pose = PoseAnalyzer().analyze(VIDEO_PHYSIO, PATIENT_ID)
    display(findings_table(pose.findings))
    rep = generate_video_report(PATIENT_ID, VIDEO_PHYSIO, pose.findings)
    display(HTML(rep.read_text(encoding="utf-8")))
else:
    print("Etapa pulada: defina VIDEO_PHYSIO ou VIDEO_SURGERY (seção 3).")

## 6. Pipeline completo (`main`): análise → fusão multimodal → alerta

Executa `src.pipeline.run`, o mesmo código usado por `python -m src.pipeline`.

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
from src.pipeline import run

video = VIDEO_SURGERY or VIDEO_PHYSIO
video_type = "surgery" if VIDEO_SURGERY else "physio"

assessment = run(PATIENT_ID, video=video, audio=AUDIO_FILE, vitals=VITALS_CSV,
                 prescriptions=PRESCRIPTIONS_CSV, dispatch=DISPATCH_ALERTS, video_type=video_type)

COLORS = {"baixa": "#2e7d32", "media": "#f9a825", "alta": "#ef6c00", "critica": "#c62828"}
mods = ", ".join(sorted({f.modality.value for f in assessment.findings})) or "nenhuma"
display(HTML(
    f"<div style='padding:12px 16px;border-radius:8px;background:{COLORS[assessment.severity.value]};"
    f"color:white;font-size:18px'>Paciente {assessment.patient_id} — risco "
    f"<b>{assessment.severity.value.upper()}</b> (score {assessment.score:.2f}) · modalidades: {mods}</div>"))
display(findings_table(assessment.findings).head(12))

In [ ]:
# Alertas gravados (o mesmo registro que o painel e o Event Grid recebem)
if settings.alerts_path.exists():
    alerts = [json.loads(l) for l in settings.alerts_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    display(pd.DataFrame(alerts)[["timestamp", "patient_id", "severity", "score", "modalities"]].tail(10))
else:
    print("Nenhum alerta gravado (severidade abaixo do mínimo ou DISPATCH_ALERTS=False).")

## 7. Salvar resultados no Google Drive

O disco do Colab é apagado ao fim da sessão. Copie alertas, relatórios, métricas e pesos para o Drive.

In [ ]:
SAVE_TO_DRIVE = False
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/resultados_tech_challenge"

if SAVE_TO_DRIVE and IN_COLAB:
    if not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    for src_dir in ["data/alerts", "data/reports", "runs"]:
        if os.path.isdir(src_dir):
            shutil.copytree(src_dir, os.path.join(DRIVE_RESULTS_DIR, os.path.basename(src_dir)),
                            dirs_exist_ok=True)
    print("Resultados copiados para", DRIVE_RESULTS_DIR)
else:
    print("Nada copiado (SAVE_TO_DRIVE=False ou fora do Colab).")